# Decision Planner Comparison for IDS

This notebook compares decision planning strategies for intrusion response in an IDS pipeline.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Install Dependencies

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

## 3) Load Dataset

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG_SEED = 42
np.random.seed(RNG_SEED)

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print('Files:', os.listdir(path))

X_test = np.load(os.path.join(path, 'X_test.npy'))
y_test = np.load(os.path.join(path, 'y_test.npy')).reshape(-1)

if X_test.ndim == 3 and X_test.shape[-1] == 1:
    X_test = X_test[..., 0]

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

## 4) Generate Sample Risk Scores

In [ ]:
def generate_samples(X, y, n_samples=600, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(y), size=min(n_samples, len(y)), replace=False)
    attack_map = {0: 'BENIGN', 1: 'DOS', 2: 'PORTSCAN', 3: 'BRUTEFORCE', 4: 'DDOS'}

    rows = []
    for i in idx:
        y_i = int(y[i])
        attack_type = attack_map.get(y_i, f'ATTACK_{y_i}')

        signal = float(np.clip(np.mean(np.abs(X[i])), 0.0, 1.0))
        risk_score = float(np.clip(0.55 * signal + 0.35 * (y_i > 0) + rng.normal(0, 0.1), 0.0, 1.0))

        if risk_score > 0.85:
            target_action = 'BLOCK_IP'
        elif risk_score > 0.65:
            target_action = 'RATE_LIMIT'
        elif risk_score > 0.45 and attack_type != 'BENIGN':
            target_action = 'ALERT_ADMIN'
        else:
            target_action = 'ALLOW'

        rows.append({
            'risk_score': risk_score,
            'attack_type': attack_type,
            'target_action': target_action,
            'is_benign': attack_type == 'BENIGN'
        })

    return pd.DataFrame(rows)

samples_df = generate_samples(X_test, y_test, n_samples=600, seed=RNG_SEED)
samples_df.head()

## 5) Implement Decision Strategies

In [ ]:
ACTIONS = ['BLOCK_IP', 'RATE_LIMIT', 'ALERT_ADMIN', 'ALLOW']

def risk_threshold_policy(risk_score, attack_type):
    del attack_type
    if risk_score > 0.85:
        return 'BLOCK_IP'
    if risk_score > 0.65:
        return 'RATE_LIMIT'
    if risk_score > 0.45:
        return 'ALERT_ADMIN'
    return 'ALLOW'

def rule_based_policy(risk_score, attack_type):
    at = attack_type.upper()
    if 'DDOS' in at or 'DOS' in at:
        return 'BLOCK_IP' if risk_score > 0.6 else 'RATE_LIMIT'
    if 'SCAN' in at or 'BRUTE' in at:
        return 'RATE_LIMIT' if risk_score > 0.55 else 'ALERT_ADMIN'
    if risk_score > 0.7:
        return 'ALERT_ADMIN'
    return 'ALLOW'

def probabilistic_planner(risk_score, attack_type, rng):
    del attack_type
    r = float(np.clip(risk_score, 0.0, 1.0))
    probs = np.array([
        0.05 + 0.70 * r,
        0.15 + 0.35 * r,
        0.35 - 0.10 * r,
        0.45 - 0.95 * r
    ], dtype=np.float64)
    probs = np.clip(probs, 1e-6, None)
    probs = probs / probs.sum()
    return rng.choice(ACTIONS, p=probs)

def rl_agent_simulated(risk_score, attack_type):
    del attack_type
    # States: low, medium, high, critical
    if risk_score > 0.8:
        s = 3
    elif risk_score > 0.6:
        s = 2
    elif risk_score > 0.4:
        s = 1
    else:
        s = 0

    # Q-table (simulated policy)
    q = np.array([
        [0.10, 0.15, 0.30, 0.70],
        [0.25, 0.45, 0.55, 0.20],
        [0.70, 0.65, 0.50, 0.10],
        [0.95, 0.70, 0.35, 0.05],
    ])
    return ACTIONS[int(np.argmax(q[s]))]

def evaluate_strategy(df, name, seed=42):
    rng = np.random.default_rng(seed)
    preds = []

    t0 = time.perf_counter()
    for _, row in df.iterrows():
        risk = float(row['risk_score'])
        attack = row['attack_type']

        if name == 'Risk Threshold Policy':
            action = risk_threshold_policy(risk, attack)
        elif name == 'Rule Based Policy':
            action = rule_based_policy(risk, attack)
        elif name == 'Probabilistic Decision Planner':
            action = probabilistic_planner(risk, attack, rng)
        elif name == 'Reinforcement Learning Agent (Simulated)':
            action = rl_agent_simulated(risk, attack)
        else:
            raise ValueError('Unknown method')

        preds.append(action)

    total = time.perf_counter() - t0
    latency = total / max(len(df), 1)

    decision_accuracy = float(np.mean(np.array(preds) == df['target_action'].values))

    benign_mask = df['is_benign'].values
    benign_preds = np.array(preds)[benign_mask]
    if len(benign_preds) == 0:
        false_mitigation_rate = 0.0
    else:
        false_mitigation_rate = float(np.mean(benign_preds != 'ALLOW'))

    return {
        'Method': name,
        'DecisionAccuracy': decision_accuracy,
        'FalseMitigationRate': false_mitigation_rate,
        'DecisionLatencySec': latency
    }

methods = [
    'Risk Threshold Policy',
    'Rule Based Policy',
    'Probabilistic Decision Planner',
    'Reinforcement Learning Agent (Simulated)'
]

rows = [evaluate_strategy(samples_df, m, seed=RNG_SEED) for m in methods]
results_df = pd.DataFrame(rows)
results_df

## Comparison Table

In [ ]:
results_df.style.format({
    'DecisionAccuracy': '{:.4f}',
    'FalseMitigationRate': '{:.4f}',
    'DecisionLatencySec': '{:.6f}'
})

## Bar Charts

In [ ]:
sns.set(style='whitegrid')

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x='Method', y='DecisionAccuracy', palette='Greens_d')
plt.ylim(0, 1)
plt.title('Decision Accuracy Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x='Method', y='FalseMitigationRate', palette='Reds_d')
plt.ylim(0, 1)
plt.title('False Mitigation Rate Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x='Method', y='DecisionLatencySec', palette='Blues_d')
plt.title('Decision Latency Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Optional best-method summary (higher accuracy, lower false mitigation, lower latency)
rank_df = results_df.copy()
rank_df['Score'] = rank_df['DecisionAccuracy'] - 0.5 * rank_df['FalseMitigationRate'] - 0.1 * (rank_df['DecisionLatencySec'] / rank_df['DecisionLatencySec'].max())
best = rank_df.sort_values('Score', ascending=False).iloc[0]
print('Best method:', best['Method'])
print(best[['DecisionAccuracy', 'FalseMitigationRate', 'DecisionLatencySec', 'Score']].to_string())